In [11]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [12]:
def get_weather(city):
    return f"The weather in {city} is sunny."

In [13]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather information for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [14]:
import json

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "What is the weather in Chennai?"
        }
    ],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

if message.tool_calls:
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "get_weather":
            result = get_weather(**arguments)
            print("Function result:", result)
else:
    print(message.content)

Function result: The weather in Chennai is sunny.


In [20]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

llm = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

response = llm.invoke("Explain function calling in one sentence.")
print(response.content)

Function calling is the act of invoking a defined function by name, supplying any required arguments, and executing its code to produce a result.


In [21]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

llm = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [22]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [23]:
llm_with_tools = llm.bind_tools([multiply])

response = llm_with_tools.invoke(
    "What is 10 multiplied by 20?"
)

print(response.tool_calls)

[{'name': 'multiply', 'args': {'a': 10, 'b': 20}, 'id': 'fc_c42f9cdd-74df-4f01-b02e-c738d66852ea', 'type': 'tool_call'}]


In [24]:
if response.tool_calls:
    for tool_call in response.tool_calls:
        if tool_call["name"] == "multiply":
            result = multiply.invoke(tool_call["args"])
            print("Function result:", result)
else:
    print(response.content)

Function result: 200
